# Variational Autoencoders (VAE)

**Domain:** Architectures  ·  **from study list**  ·  **runnable:** yes

A refresher on the **VAE** (Kingma & Welling, 2013): an autoencoder turned into a
proper *generative* model by making the bottleneck **probabilistic** and training
it to maximize a variational lower bound on the data likelihood.

## 1. What & Why

A **VAE** is an autoencoder whose latent code is a **distribution, not a point**.
The encoder maps an input `x` to the parameters of a Gaussian `q(z|x) = N(μ, σ²)`;
you **sample** a `z` from it; the decoder maps `z` back to a reconstruction. Training
maximizes the **ELBO** — reconstruction quality *minus* a KL term that pulls every
per-input Gaussian toward a fixed **prior** `p(z) = N(0, I)`.

**The problem it solves.** A vanilla autoencoder learns a great compressor but a
*useless generator*: its latent space is full of holes and discontinuities, so a
random `z` (or a point between two encodings) decodes to garbage. The VAE's KL
regularizer forces the encoded codes to **pack smoothly around `N(0, I)`**, so the
whole neighborhood of the origin decodes to plausible data. That makes two new
things possible that a plain AE can't do:

- **Sampling** — draw `z ~ N(0, I)`, decode, and get a brand-new sample.
- **Smooth interpolation / arithmetic** in a continuous latent space.

**Reach for a VAE when** you want a generative model *with an encoder* (amortized
inference), a **smooth, structured latent space** you can interpolate or condition
on, a **principled likelihood bound**, and **stable, single-pass** training and
sampling. It's also the standard workhorse for **anomaly detection** (low ELBO =
out of distribution) and as the **latent compressor inside latent diffusion**
(Stable Diffusion's VAE).

**Don't reach for it when** you need the **sharpest** possible samples — the
Gaussian likelihood + averaging over the posterior makes VAE images famously
**blurry**. For raw fidelity, prefer a **GAN** (`gan.ipynb`) or, today, a
**diffusion model** (`diffusion-models.ipynb`).

## 2. Mental Model

A vanilla AE maps each input to a single **point**. A VAE maps it to a **fuzzy
cloud** (a Gaussian), and the KL term shrinks and recenters every cloud so the
clouds **tile the space around the origin with no gaps**:

```
                  encoder                 reparameterize             decoder
        x  ───────────────▶  μ(x), log σ²(x)  ──────▶  z = μ + σ⊙ε  ──────▶  x̂
    (image)                                          ▲   ε ~ N(0, I)        (recon)
                                                     │
                    two pressures fight over z ───────┘
      reconstruction loss : spread the clouds apart so x̂ ≈ x  (be informative)
      KL toward N(0, I)   : pull every cloud back to the origin (be regular)
```

- **Reconstruction** wants codes far apart and tight — ideally back to points, which
  is just a plain AE with all its holes.
- **KL** wants every cloud to *be* the unit Gaussian — which would erase all info.
- The **balance** is a latent space that is both *informative* and *gap-free*:
  exactly the property that lets you sample `z ~ N(0, I)` and decode something real.

The **reparameterization trick** is the load-bearing detail: writing
`z = μ + σ ⊙ ε` (with `ε ~ N(0, I)` drawn *outside* the network) moves the randomness
off the computation path, so gradients can flow through `μ` and `σ`. You can't
backprop through a raw `sample()`; you can backprop through `μ + σ ⊙ ε`.

## 3. Key Concepts

**The objective — the ELBO** (Evidence Lower BOund). We can't maximize `log p(x)`
directly, so we maximize a tractable lower bound on it:

$$
\log p(x) \;\ge\; \mathcal{L}(x) =
\underbrace{\mathbb{E}_{q(z|x)}\big[\log p(x|z)\big]}_{\text{reconstruction}}
\;-\;
\underbrace{D_{\mathrm{KL}}\!\big(q(z|x)\,\|\,p(z)\big)}_{\text{regularizer}}
$$

We **maximize the ELBO** = **minimize** `(reconstruction loss) + (KL)`. The gap
between `log p(x)` and the ELBO is exactly `KL(q(z|x) ‖ p(z|x))` — the bound is tight
when the encoder matches the true posterior.

**KL has a closed form** for two Gaussians with a unit prior `p(z)=N(0,I)`. Per
latent dimension:

$$
D_{\mathrm{KL}} = -\tfrac{1}{2}\sum_{j}\Big(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\Big)
$$

so no sampling is needed for the KL term — only for the reconstruction term.

| Term | What it is |
|---|---|
| **Encoder / inference net `q(z|x)`** | Outputs `μ` and `log σ²` (predict log-variance for numerical stability). "Amortized" = one network for all `x`. |
| **Decoder / generative net `p(x|z)`** | Maps a latent sample back to data (Bernoulli/Gaussian likelihood). |
| **Prior `p(z)`** | Fixed `N(0, I)`; what you sample from at generation time. |
| **Reparameterization** | `z = μ + σ ⊙ ε`, `ε ~ N(0,I)` — makes the sampling step differentiable. |
| **Reconstruction term** | `−log p(x|z)`: BCE for binary/[0,1] data, MSE for Gaussian data. |
| **KL term** | Pulls `q(z|x)` toward `N(0,I)`; the closed form above. |
| **β-VAE** | Scale the KL by `β`. `β>1` → more disentangled but blurrier; `β<1` → sharper but less regular. |
| **Posterior collapse** | KL → 0: the decoder ignores `z` and `q(z|x)=p(z)`; the latent carries no information. |

**Variants to recognize:** **β-VAE** (disentanglement), **Conditional VAE (CVAE)**
(condition encoder/decoder on a label), **VQ-VAE** (discrete codebook latent —
powers many modern tokenizers), and the **VAE in latent diffusion** (compress to a
latent, then run diffusion there).

## 4. Setup

The examples use **PyTorch** and **scikit-learn** (only for its tiny built-in
`load_digits` dataset — 1797 8×8 images, no download). Everything runs on CPU in a
few seconds. The optional final cell sketches a convolutional MNIST VAE and is
gated behind an env var so the notebook always executes top-to-bottom.

In [1]:
# %pip install torch scikit-learn numpy
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.datasets import load_digits

torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__, "| numpy", np.__version__)

torch 2.12.1 | numpy 2.4.6


## 5. Worked Examples

### Example 1 — A complete VAE on 8×8 digits

The smallest VAE that does everything a VAE is for. We compress 64-pixel digit
images down to a **2-D latent**, train by minimizing `BCE_recon + KL`, and watch
both terms move: reconstruction *falls* (the decoder gets better) while KL *rises
off zero* (the codes spread out to carry information, but stay reined in by the
prior). At the end we draw `z ~ N(0, I)` and decode — the payoff a plain AE can't
give you.

In [2]:
X = load_digits().data / 16.0                      # 1797 x 64, scaled to [0, 1]
X = torch.tensor(X, dtype=torch.float32)
N, D, Z = X.shape[0], X.shape[1], 2                 # 2-D latent so we can sample it

class VAE(nn.Module):
    def __init__(self, d=D, z=Z, h=64):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d, h), nn.ReLU())
        self.mu = nn.Linear(h, z)
        self.logvar = nn.Linear(h, z)               # predict log-variance, not variance
        self.dec = nn.Sequential(nn.Linear(z, h), nn.ReLU(), nn.Linear(h, d))

    def encode(self, x):
        h = self.enc(x)
        return self.mu(h), self.logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)                 # noise drawn OUTSIDE the graph
        return mu + std * eps                       # differentiable in mu, logvar

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.dec(z), mu, logvar              # decoder returns logits

def elbo_loss(x, logits, mu, logvar):
    # reconstruction: Bernoulli NLL summed over pixels, averaged over the batch
    recon = F.binary_cross_entropy_with_logits(logits, x, reduction="sum") / x.shape[0]
    # closed-form KL( N(mu, sigma^2) || N(0, I) ), per the formula in section 3
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.shape[0]
    return recon + kl, recon, kl

model = VAE()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(60):
    perm = torch.randperm(N)
    for i in range(0, N, 256):
        xb = X[perm[i:i + 256]]
        logits, mu, logvar = model(xb)
        loss, recon, kl = elbo_loss(xb, logits, mu, logvar)
        opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 15 == 0 or epoch == 59:
        print(f"epoch {epoch:>2}  ELBO_loss={loss.item():6.2f}  "
              f"recon={recon.item():6.2f}  KL={kl.item():5.2f}")

print("\nKL climbed off 0 (codes carry info) but stayed small (prior keeps them tidy).")

epoch  0  ELBO_loss= 43.02  recon= 42.97  KL= 0.05
epoch 15  ELBO_loss= 28.15  recon= 26.89  KL= 1.26
epoch 30  ELBO_loss= 29.26  recon= 28.42  KL= 0.84
epoch 45  ELBO_loss= 25.20  recon= 23.55  KL= 1.65


epoch 59  ELBO_loss= 25.77  recon= 24.48  KL= 1.30

KL climbed off 0 (codes carry info) but stayed small (prior keeps them tidy).


In [3]:
# The payoff: decode fresh samples drawn straight from the prior N(0, I).
model.eval()
with torch.no_grad():
    z = torch.randn(5, Z)                            # never came from any real image
    gen = torch.sigmoid(model.dec(z)).reshape(5, 8, 8)

chars = np.array(list(" .:-=+*#%@"))                 # ASCII ramp for 10 intensity bins
for k in range(5):
    img = gen[k].numpy()
    print(f"prior sample {k}:")
    for row in img:
        print("   " + "".join(chars[np.clip((row * 9).astype(int), 0, 9)]))
print("Each is a NEW digit-like glyph synthesized from random latent noise.")

prior sample 0:
     :**:  
    .+*+=  
    .*-:-. 
    .*==-. 
    .===+. 
    .=--+: 
     ++=*- 
     :*#=. 
prior sample 1:
     =##=  
    .****. 
    .=-*=  
     -*#-  
     :*#:  
     :++:  
     =*+-  
     =#+:  
prior sample 2:
     =#+.  
    .#++=  
    :+:==  
    .==+=  
     :-=+. 
     :..+= 
     +=-#+ 
     =#%*: 
prior sample 3:
      +*:  
     -#+:. 
     *+.-: 
    :#-:=: 
    -#=+*: 
    :*=++: 
     =++=  
     .+#-  
prior sample 4:
      =#=. 
     :#+=: 
     =+-=: 
    .*=**: 
    :###*. 
    .=*#=  
     .+#.  
      +=   
Each is a NEW digit-like glyph synthesized from random latent noise.


### Example 2 — The reparameterization trick & the closed-form KL

Two claims from section 3, made concrete. **(a)** Sampling with
`z = μ + σ⊙ε` is differentiable in `μ` and `σ`, whereas a raw `Normal(μ,σ).sample()`
is a dead end for gradients. **(b)** The closed-form KL matches a Monte-Carlo
estimate — which is why we never sample for the KL term.

In [4]:
mu = torch.tensor([0.7], requires_grad=True)
logvar = torch.tensor([0.0], requires_grad=True)   # sigma = 1
std = torch.exp(0.5 * logvar)

# (a) reparameterized sample -> gradient flows back to mu and logvar
eps = torch.randn(10000)
z = mu + std * eps
z.mean().backward()
print("reparameterized:  d mean(z)/d mu     =", round(mu.grad.item(), 4), "(≈1, as it should)")

# A direct .sample() detaches: there is simply no grad_fn to back-propagate through.
direct = torch.distributions.Normal(mu.detach(), std.detach()).sample((10000,))
print("direct .sample():  requires_grad     =", direct.requires_grad, "(no gradient path)")

# (b) closed-form KL vs Monte-Carlo estimate of E_q[log q - log p]
m, lv = torch.tensor(0.7), torch.tensor(0.5)
s = torch.exp(0.5 * lv)
kl_closed = (-0.5 * (1 + lv - m.pow(2) - lv.exp())).item()
q, p = torch.distributions.Normal(m, s), torch.distributions.Normal(0.0, 1.0)
zs = q.sample((200000,))
kl_mc = (q.log_prob(zs) - p.log_prob(zs)).mean().item()
print(f"\nKL closed-form = {kl_closed:.4f}   KL Monte-Carlo = {kl_mc:.4f}   (they agree)")

reparameterized:  d mean(z)/d mu     = 1.0 (≈1, as it should)
direct .sample():  requires_grad     = False (no gradient path)

KL closed-form = 0.3194   KL Monte-Carlo = 0.3188   (they agree)


### Example 3 — A convolutional MNIST VAE (optional, gated)

A real image VAE wants a conv encoder/decoder and the full MNIST dataset, so this
cell only runs when you opt in. It shows the *shape* of the scale-up — strided
convs down to `(μ, logσ²)`, transposed convs back up — while the loss and training
loop stay identical to Example 1. The notebook still executes without it.

In [5]:
import os

if os.getenv("RUN_MNIST_VAE"):
    # Requires: pip install torchvision ; downloads MNIST (~12 MB); slow on CPU.
    z_dim = 16
    encoder = nn.Sequential(
        nn.Conv2d(1, 32, 3, 2, 1), nn.ReLU(),        # 28 -> 14
        nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),       # 14 -> 7
        nn.Flatten(),
    )
    enc_mu, enc_logvar = nn.Linear(64 * 7 * 7, z_dim), nn.Linear(64 * 7 * 7, z_dim)
    decoder = nn.Sequential(
        nn.Linear(z_dim, 64 * 7 * 7), nn.ReLU(),
        nn.Unflatten(1, (64, 7, 7)),
        nn.ConvTranspose2d(64, 32, 3, 2, 1, output_padding=1), nn.ReLU(),   # 7 -> 14
        nn.ConvTranspose2d(32, 1, 3, 2, 1, output_padding=1),               # 14 -> 28 (logits)
    )
    x = torch.randn(4, 1, 28, 28)
    h = encoder(x)
    mu, logvar = enc_mu(h), enc_logvar(h)
    z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
    print("latent z shape :", tuple(z.shape))
    print("recon shape    :", tuple(decoder(z).shape))
    print("Wire up the SAME elbo_loss + Adam loop from Example 1 over real MNIST batches.")
else:
    print("Set RUN_MNIST_VAE=1 (and `pip install torchvision`) to build a conv VAE on MNIST.")
    print("Shape of the scale-up:")
    print("  encoder: 1x28x28 --stride-2 convs--> flatten --> (mu, logvar)  in R^16")
    print("  decoder: z(16) --linear--> 64x7x7 --transposed convs--> 1x28x28 logits")
    print("  loss/loop: identical recon(BCE) + KL objective from Example 1.")

Set RUN_MNIST_VAE=1 (and `pip install torchvision`) to build a conv VAE on MNIST.
Shape of the scale-up:
  encoder: 1x28x28 --stride-2 convs--> flatten --> (mu, logvar)  in R^16
  decoder: z(16) --linear--> 64x7x7 --transposed convs--> 1x28x28 logits
  loss/loop: identical recon(BCE) + KL objective from Example 1.


## 6. Gotchas & Pitfalls

- **Predict `log σ²`, not `σ`.** Outputting log-variance lets the net produce any
  real number while `σ = exp(½ logσ²)` stays positive, and it keeps the KL stable.
  Outputting `σ` directly invites negatives / NaNs.
- **Reparameterize — never backprop through `.sample()`.** `z = μ + σ⊙ε` with `ε`
  drawn outside the graph is the whole trick (Example 2). A raw `dist.sample()`
  has no gradient path and silently kills learning of `μ`/`σ`.
- **Posterior collapse.** With a powerful (e.g. autoregressive) decoder the model
  drives `KL → 0`, ignores `z`, and the latent becomes useless. Mitigations:
  **KL annealing** (ramp the KL weight up from 0), **free bits** (floor the KL per
  dim), or a weaker decoder.
- **Match the reconstruction loss to the data.** **BCE** for binary/[0,1] pixels,
  **MSE / Gaussian NLL** for real-valued data. The wrong likelihood quietly
  reweights recon-vs-KL and wrecks samples.
- **Watch the recon/KL reduction.** Sum the reconstruction over pixels (a *total*
  log-likelihood), then divide by batch size — don't `mean` over pixels, or the
  KL will dwarf the reconstruction and you'll get **posterior collapse to a
  blurry mean**. The recon–KL ratio *is* `β`, whether you meant it or not.
- **`β` is a real knob (β-VAE).** `β > 1` → more disentangled, blurrier, more
  collapse risk; `β < 1` → sharper but a leakier latent you can't cleanly sample.
- **Blurriness is structural, not a bug.** A pixel-wise Gaussian/Bernoulli decoder
  averages over the posterior, so VAE samples are inherently soft. If you need
  crisp images, switch model class (GAN/diffusion) or compose (VQ-VAE + prior,
  VAE-in-latent-diffusion) — don't just tune `β`.
- **Sample the *prior* to generate, the *posterior* to reconstruct.** Generation
  draws `z ~ N(0, I)`; reconstruction encodes a real `x` to `q(z|x)`. Mixing them
  up (e.g. sampling `μ(x)` and calling it "generation") is a common confusion.

## 7. When to Use vs Alternatives

| Option | Pick it when… | Trade-off vs VAE |
|---|---|---|
| **Plain autoencoder** | You only need compression / denoising, never sampling. | Simpler, sharper recon, but a **holey latent** you can't sample. See `autoencoders.ipynb`. |
| **GAN** | You want **sharp** samples and don't need an encoder or likelihood. | Crisper images, but **no encoder**, unstable training, mode collapse. See `gan.ipynb`. |
| **Diffusion model** | You want **best-in-class fidelity & diversity**. | Higher quality + stable, but **slow multi-step sampling** and no compact latent (unless latent-diffusion, which *uses* a VAE). See `diffusion-models.ipynb`. |
| **Normalizing flow** | You need an **exact** likelihood and invertible mapping. | Exact `p(x)` (vs the VAE's *bound*), but architecturally constrained. See `normalizing-flows.ipynb`. |
| **VQ-VAE** | You want a **discrete** latent / tokenizer for a downstream prior. | Discrete codes feed transformers/diffusion, but adds a codebook + second-stage prior. |

**Rule of thumb (2026):** use a **VAE** when you specifically need an **encoder +
smooth latent space** — representation learning, anomaly detection, conditional
generation, or as the latent compressor inside a larger system. For standalone
**image quality**, reach for **diffusion** first and **GAN** when sampling speed
matters. The VAE's enduring role in 2026 is less "SOTA generator" and more
"the well-behaved latent space everything else is built on top of." Closely
related notebooks: `autoencoders.ipynb`, `gan.ipynb`, `diffusion-models.ipynb`,
`normalizing-flows.ipynb`.

## 8. Resources

- **Auto-Encoding Variational Bayes** — Kingma & Welling, 2013 (the original VAE
  paper, introduces the reparameterization trick): https://arxiv.org/abs/1312.6114
- **An Introduction to Variational Autoencoders** — Kingma & Welling, 2019, the
  book-length tutorial by the authors: https://arxiv.org/abs/1906.02691
- **β-VAE** — Higgins et al., 2017, on disentanglement via the KL weight:
  https://openreview.net/forum?id=Sy2fzU9gl
- **Neural Discrete Representation Learning (VQ-VAE)** — van den Oord et al., 2017:
  https://arxiv.org/abs/1711.00937
- **From Autoencoder to Beta-VAE** — Lilian Weng's deep-dive on the math:
  https://lilianweng.github.io/posts/2018-08-12-vae/
- **PyTorch VAE example** — a complete, runnable reference implementation:
  https://github.com/pytorch/examples/tree/main/vae